# 1. Dataset de entrenamiento y validación propia

**Fuentes**
1. **Spider** (train + others + dev), convertido de SQLite a MySQL 8.4.10 y con las preguntas traducidas al español
   (Helsinki-NLP/opus-mt-en-es, protegiendo los literales de la consulta con marcadores).
2. **12 esquemas propios** (5–8 tablas, nombres en español/inglés, completos o abreviados, con `COMMENT`),
   con pares pregunta–SQL generados con un LLM. Cubren lo que Spider no tiene: funciones de ventana, CTE, fechas de MySQL.

Solo se aceptan pares cuya consulta se ejecuta sin error en MySQL 8.4.10 (`ONLY_FULL_GROUP_BY` activo).
La validación propia usa 2 esquemas que **no** aparecen en entrenamiento (partición por esquema).

In [1]:
import json, os, sys
from pathlib import Path
import pandas as pd
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ); sys.path.insert(0, "scripts")
PASOS_PESADOS = False  # True para regenerar todo (requiere MySQL 8.4.10 en Docker y GPU)
pd.set_option("display.max_colwidth", 120)

In [2]:
if PASOS_PESADOS:
    !python scripts/spider_a_mysql.py C:/spider_dl/spider_data
    !python scripts/traducir_spider.py C:/spider_dl/spider_data
    !python scripts/construir_dataset.py
resumen = json.loads(Path("datos/resumen_dataset.json").read_text(encoding="utf8"))
{k: v for k, v in resumen.items() if k != "tablas_validacion"}

{'pares_spider_validos': 8828,
 'pares_propios_unicos': 1330,
 'esquemas_propios': 18,
 'ejemplos_entrenamiento_con_repeticion': 12620,
 'ejemplos_dev_loss': 238,
 'preguntas_validacion': 40,
 'descartes_por_motivo': {'resultado_distinto_sqlite': 408,
  'error_mysql': 191,
  'traduccion_perdio_valor': 83,
  'esquema_demasiado_largo': 183},
 'total_descartes': 865,
 'fuentes': {'propio': 3990,
  'spider_train': 6035,
  'spider_dev': 950,
  'spider_others': 1645}}

## Descartes y motivo

In [3]:
desc = pd.read_csv("datos/descartes.csv")
print(desc.motivo.value_counts().to_string())
desc.groupby("motivo").head(2)[["motivo", "question", "query_sqlite", "detalle"]]

motivo
resultado_distinto_sqlite    408
error_mysql                  191
esquema_demasiado_largo      183
traduccion_perdio_valor       83


,motivo,question,query_sqlite,detalle
0,resultado_distinto_sqlite,Which rank has the smallest number of faculty members?,SELECT rank FROM Faculty GROUP BY rank ORDER BY count(*) ASC LIMIT 1,NaN
1,resultado_distinto_sqlite,Find the faculty rank that has the least members.,SELECT rank FROM Faculty GROUP BY rank ORDER BY count(*) ASC LIMIT 1,NaN
8,error_mysql,"Show the apartment type code that has the largest number of total rooms, together with the number of bathrooms and n...","SELECT apt_type_code , bathroom_count , bedroom_count FROM Apartments GROUP BY apt_type_code ORDER BY sum(room_cou...","(1055, ""Expression #2 of SELECT list is not in GROUP BY clause and contains nonaggregated column 'sp_apartment_renta..."
9,error_mysql,"Which apartment type has the largest number of total rooms? Return the apartment type code, its number of bathrooms ...","SELECT apt_type_code , bathroom_count , bedroom_count FROM Apartments GROUP BY apt_type_code ORDER BY sum(room_cou...","(1055, ""Expression #2 of SELECT list is not in GROUP BY clause and contains nonaggregated column 'sp_apartment_renta..."
599,traduccion_perdio_valor,How many female Professors do we have?,"SELECT count(*) FROM Faculty WHERE Sex = 'F' AND Rank = ""Professor""",¿Cuántas profesoras tenemos?
600,traduccion_perdio_valor,Count the number of female Professors we have.,"SELECT count(*) FROM Faculty WHERE Sex = 'F' AND Rank = ""Professor""",Cuenta el número de profesoras que tenemos.
620,esquema_demasiado_largo,Which assets have 2 parts and have less than 2 fault logs? List the asset id and detail.,"SELECT T1.asset_id , T1.asset_details FROM Assets AS T1 JOIN Asset_Parts AS T2 ON T1.asset_id = T2.asset_id GROUP...",¿Qué activos tienen 2 partes y tienen menos de 2 registros de fallas?
621,esquema_demasiado_largo,Which third party companies have at least 2 maintenance engineers or have at least 2 maintenance contracts? List the...,"SELECT T1.company_id , T1.company_name FROM Third_Party_Companies AS T1 JOIN Maintenance_Engineers AS T2 ON T1.comp...",¿Qué empresas de terceros tienen al menos 2 ingenieros de mantenimiento o tienen al menos 2 contratos de mantenimiento?


- **resultado_distinto_sqlite**: la consulta corre en MySQL pero devuelve otra cosa que en SQLite (división entera, comparación de
  cadenas sin distinguir mayúsculas, `LIMIT` con empates…). Se descarta para no enseñar consultas cuyo significado cambió.
- **error_mysql**: falla en MySQL 8.4.10, típicamente por `ONLY_FULL_GROUP_BY`, palabras reservadas o funciones de SQLite.
- **traduccion_perdio_valor**: el traductor alteró un literal que la consulta necesita (p. ej. un nombre propio).
- **esquema_demasiado_largo**: prompt + respuesta > 2048 tokens.

## Ejemplos de entrenamiento

In [4]:
train = pd.read_json("datos/entrenamiento.jsonl", lines=True)
print(train.fuente.value_counts().to_string())
train.groupby("fuente").head(3)[["fuente", "db", "pregunta", "sql"]]

fuente
spider_train     6035
propio           3990
spider_others    1645
spider_dev        950


,fuente,db,pregunta,sql
0,propio,own_cine,¿Qué películas nunca han tenido ninguna función? Devuelve título y fecha de estreno.,"SELECT p.tit, p.f_estr FROM pel p LEFT JOIN func f ON f.id_pel = p.id_pel WHERE f.id_func IS NULL"
1,propio,own_gimnasio,"En cada sala, ¿cuál fue la clase con más asistentes? Muestra sala, identificador y nombre de la clase y número de as...","WITH a AS (SELECT c.room, c.cls_id, c.cls_name, COUNT(*) AS asistentes FROM cls c JOIN cls_enroll e ON e.cls_id = c...."
2,spider_train,sp_workshop_paper,"Para cada presentación, encuentre su autor y el resultado de la aceptación.","SELECT T2.Author, T1.Result FROM Acceptance AS T1 JOIN submission AS T2 ON T1.Submission_ID = T2.Submission_ID"
3,propio,own_biblioteca,Lista el nombre y el apellido de los autores de nacionalidad española.,"SELECT fnm, lnm FROM auth WHERE ntn = 'España'"
6,spider_train,sp_customers_and_addresses,Devuelve el número de clientes que tienen al menos un pedido con estatus Cancelled.,SELECT COUNT(DISTINCT customer_id) FROM Customer_Orders WHERE order_status = 'Cancelled'
7,spider_train,sp_music_4,¿Cuál es la categoría de festivales de música con resultado Awarded?,SELECT Category FROM music_festival WHERE RESULT = 'Awarded'
9,spider_dev,sp_car_1,¿Cuál es la potencia del coche con la mayor aceleración?,SELECT T1.horsepower FROM cars_data AS T1 ORDER BY T1.accelerate DESC LIMIT 1
10,spider_others,sp_geo,¿Cuáles son las principales ciudades de los estados a través de los cuales corre el río principal en virginia,SELECT city_name FROM city WHERE population > 150000 AND state_name IN (SELECT traverse FROM river WHERE river_name ...
23,spider_others,sp_geo,lo que es la montaña más alta en nosotros,SELECT mountain_name FROM mountain WHERE mountain_altitude = (SELECT MAX(mountain_altitude) FROM mountain)
25,spider_others,sp_academic,"Devuélveme la página de inicio de "" University of Michigan "" .",SELECT homepage FROM organization WHERE name = 'University of Michigan'


In [5]:
propios = train[train.fuente == "propio"].drop_duplicates(["db", "pregunta"])
pd.crosstab(propios.db, propios.nivel, margins=True)

nivel,dificil,facil,media,All
db,,,,
own_aerolinea,22,21,28,71
own_banco,21,23,29,73
own_biblioteca,21,21,28,70
own_cine,33,16,32,81
own_delivery_comida,22,21,28,71
own_gimnasio,33,16,32,81
own_hospital,21,21,28,70
own_hotel,20,20,26,66
own_inmobiliaria,33,16,32,81


In [6]:
print(train[train.fuente == "propio"].iloc[0].esquema[:1500])

CREATE TABLE `bol` (
  `id_bol` int NOT NULL COMMENT 'Identificador del boleto',
  `id_func` int NOT NULL COMMENT 'Función para la que se emitió',
  `id_cli` int DEFAULT NULL COMMENT 'Cliente socio que compró; NULL si fue venta sin registro',
  `f_compra` datetime NOT NULL COMMENT 'Fecha y hora de compra',
  `asiento` varchar(4) NOT NULL COMMENT 'Butaca asignada (fila y número, ej. F12)',
  `prec_pag` decimal(8,2) NOT NULL COMMENT 'Precio efectivamente pagado en pesos (puede incluir descuento)',
  `canal` char(3) NOT NULL COMMENT 'Canal de venta: TAQ=taquilla, WEB=sitio web, APP=aplicación móvil',
  `est` char(3) NOT NULL COMMENT 'Estado del boleto: VAL=vigente, USA=usado, REE=reembolsado',
  PRIMARY KEY (`id_bol`),
  KEY `id_func` (`id_func`),
  KEY `id_cli` (`id_cli`),
  CONSTRAINT `bol_ibfk_1` FOREIGN KEY (`id_func`) REFERENCES `func` (`id_func`),
  CONSTRAINT `bol_ibfk_2` FOREIGN KEY (`id_cli`) REFERENCES `cli` (`id_cli`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_0900

## Validación propia (40 preguntas, 2 esquemas no vistos)

In [7]:
val = pd.read_json("datos/validacion_propia.jsonl", lines=True)
display(pd.crosstab(val.db, val.nivel, margins=True))
assert not set(val.db.str.replace("val_", "own_")) & set(train.db), "esquema de validación presente en entrenamiento"
val[["db", "nivel", "pregunta", "sql"]]

nivel,dificil,facil,media,All
db,,,,
val_conference_events,6,6,8,20
val_taller_mecanico,6,6,8,20
All,12,12,16,40


,db,nivel,pregunta,sql
0,val_conference_events,facil,"Lista el nombre, la ciudad y el aforo de los recintos con capacidad para más de 5000 personas.","SELECT venue_name, city, capacity FROM venue WHERE capacity > 5000"
1,val_conference_events,facil,¿Cuántas entradas han sido reembolsadas? Devuelve solo el número.,SELECT COUNT(*) AS reembolsadas FROM ticket WHERE status = 'refunded'
2,val_conference_events,facil,Muestra el nombre completo y el país de los ponentes que no tienen una empresa registrada.,"SELECT full_name, country FROM speaker WHERE company IS NULL"
3,val_conference_events,facil,"Lista el título, la fecha de inicio y el precio base de los eventos de la categoría 'Inteligencia Artificial', orden...","SELECT title, start_dt, base_price FROM evt WHERE cat = 'Inteligencia Artificial' ORDER BY start_dt"
4,val_conference_events,facil,¿Cuál es la duración promedio en minutos de las sesiones de tipo taller (workshop)? Devuelve el valor redondeado a 2...,"SELECT ROUND(AVG(dur_min), 2) AS duracion_promedio FROM session WHERE sess_type = 'workshop'"
5,val_conference_events,facil,"Muestra, sin repetir, los códigos de descuento que se han utilizado en la compra de entradas.",SELECT DISTINCT disc_code FROM ticket WHERE disc_code IS NOT NULL
6,val_conference_events,media,"Para cada evento, muestra su título y la cantidad de entradas pagadas (sin contar las reembolsadas). Los eventos que...","SELECT e.title, COUNT(t.ticket_id) AS entradas_pagadas FROM evt e LEFT JOIN ticket t ON t.evt_id = e.evt_id AND t.st..."
7,val_conference_events,media,"Considerando solo las entradas pagadas, muestra por cada tipo de entrada el tipo, la cantidad vendida y el ingreso t...","SELECT tkt_type, COUNT(*) AS cantidad, ROUND(SUM(price_paid), 2) AS ingreso_total FROM ticket WHERE status = 'paid' ..."
8,val_conference_events,media,¿Qué ponentes tienen a su cargo más de 2 sesiones? Devuelve el nombre completo del ponente y el número de sesiones.,"SELECT s.full_name, COUNT(*) AS num_sesiones FROM speaker s JOIN session se ON se.spk_id = s.spk_id GROUP BY s.spk_i..."
9,val_conference_events,media,"Para los eventos ya realizados (completed), muestra por cada país del recinto el país, el número de eventos y el pre...","SELECT v.country, COUNT(*) AS num_eventos, ROUND(AVG(e.base_price), 2) AS precio_promedio FROM venue v JOIN evt e ON..."
